<a href="https://colab.research.google.com/github/DaniloDuque/logistic-regression/blob/main/src/tp2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP2 — Regresión Logística

## Setup

In [ ]:
import os, sys

if 'google.colab' in sys.modules:
    if not os.path.exists('logistic-regression'):
        os.system('git clone https://github.com/DaniloDuque/logistic-regression.git')
    os.chdir('logistic-regression')

sys.path.insert(0, os.path.abspath('src'))

---
# Sección 1 — Algoritmo de Regresión Logística

## 1.a — Usar el código del perceptrón como base
DOCUMENTACION EN LATEX: Explique en el reporte cada una de las modificaciones necesarias al código del perceptrón, utilizando como referencia las diferencias entre ambos algoritmos.

### Pruebas unitarias
2 pruebas unitarias (con su objetivo, entradas, salidas esperadas y resultados) para las funciones modificadas en el algoritmo del perceptron base.

---
## 1.b — Experimentos: Datos separables vs no separables

### Imports

In [ ]:
# Install spacy
!pip install spacy

# Download the Spanish model
!python -m spacy download es_core_news_lg

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import spacy
from pathlib import Path

from data_generator import generate_data
from logistic_regression import LogisticRegression
from trainer import train_with_history, compute_mae
from metrics import run_experiment, print_single_result, print_runs_table, compute_accuracy
from visualization import plot_results

STEPS = 1000
ALPHA = 0.1

FIGURES_DIR = Path(__file__).parent.parent / 'figures' if '__file__' in dir() else Path('..') / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)


### Experimento 1 — Datos linealmente separables

Dos clases generadas con `cluster_std=1.0` (grupos compactos y bien separados).
Se espera que el modelo converja rápidamente y obtenga un MAE bajo.


In [ ]:
X_train_s, X_test_s, y_train_s, y_test_s = generate_data(separable=True, n_samples=500, random_state=42)

model_s = LogisticRegression(torch.zeros(X_train_s.shape[1], 1))
train_errors_s, test_errors_s = train_with_history(
    model_s, X_train_s, y_train_s, X_test_s, y_test_s, steps=STEPS, alpha=ALPHA
)

mae_train_s = compute_mae(y_train_s, model_s.forward(X_train_s))
mae_test_s  = compute_mae(y_test_s,  model_s.forward(X_test_s))
acc_train_s = compute_accuracy(model_s, X_train_s, y_train_s)
acc_test_s  = compute_accuracy(model_s, X_test_s,  y_test_s)

print(f"Iteraciones: {STEPS}")
print_single_result('Linealmente separable', mae_train_s, mae_test_s, acc_train_s, acc_test_s)

plot_results(model_s, X_train_s, y_train_s, train_errors_s, test_errors_s,
             'Experimento 1 — Datos linealmente separables',
             output_path=FIGURES_DIR / 'experimento1.pdf')


### Experimento 2 — Datos no linealmente separables

Dos clases generadas con `cluster_std=4.0` (grupos solapados).
Se espera que el modelo tenga dificultades y obtenga un MAE más alto.


In [ ]:
X_train_ns, X_test_ns, y_train_ns, y_test_ns = generate_data(separable=False, n_samples=500, random_state=42)

model_ns = LogisticRegression(torch.zeros(X_train_ns.shape[1], 1))
train_errors_ns, test_errors_ns = train_with_history(
    model_ns, X_train_ns, y_train_ns, X_test_ns, y_test_ns, steps=STEPS, alpha=ALPHA
)

mae_train_ns = compute_mae(y_train_ns, model_ns.forward(X_train_ns))
mae_test_ns  = compute_mae(y_test_ns,  model_ns.forward(X_test_ns))
acc_train_ns = compute_accuracy(model_ns, X_train_ns, y_train_ns)
acc_test_ns  = compute_accuracy(model_ns, X_test_ns,  y_test_ns)

print(f"Iteraciones: {STEPS}")
print_single_result('No linealmente separable', mae_train_ns, mae_test_ns, acc_train_ns, acc_test_ns)

plot_results(model_ns, X_train_ns, y_train_ns, train_errors_ns, test_errors_ns,
             'Experimento 2 — Datos no linealmente separables',
             output_path=FIGURES_DIR / 'experimento2.pdf')


### 1.2 — 10 ejecuciones: MAE medio, precisión, mínimo y máximo


In [ ]:
results_s  = run_experiment(separable=True,  steps=STEPS, alpha=ALPHA)
results_ns = run_experiment(separable=False, steps=STEPS, alpha=ALPHA)

print_runs_table(results_s, results_ns)

---
# Sección 2 — Regresión Logística y LLMs para clasificación de textos

## Pre requisitos


In [ ]:
# Prerequisitos
import torch

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from dataset import load_feina
from logistic_regression import LogisticRegression

import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

#!pip install spacy
#!python -m spacy download es_core_news_lg
#import spacy

#!pip install simplemma
#import simplemma

# Cargar dataset de FEINA
df, all_texts, all_labels = load_feina()
#    df         : pd.DataFrame  — dataframe completo
#    texts      : list[str]     — textos (Segment + Proposal concatenados)
#    labels     : list[int]     — 1 = complejo (Segment), 0 = simple (Proposal)


STEPS_LR = 3000


#Sección 2.1 - Regresión logística con la TFIDF.

In [ ]:
# Preprocesamiento del texto

def preprocess_text(document):
    tokeniser = RegexpTokenizer(r'\w+')
    tokens = tokeniser.tokenize(document)

    lemmatiser = WordNetLemmatizer()
    lemmas = [lemmatiser.lemmatize(token.lower(), pos='v') for token in tokens]

    keywords = [lemma for lemma in lemmas if lemma not in stopwords.words('spanish')]
    return keywords

print("── a) Examen cualitativo del preprocesamiento ──")
for text in all_texts[:3]:
    print(f"  ORIGINAL:  {text}")
    print(f"  PROCESADO: {preprocess_text(text)}")
    print()

In [ ]:
# Construcción de descriptores TF-IDF
def display_tfidfs(X_vectorised):
    df_sparse = pd.DataFrame.sparse.from_spmatrix(X_vectorised)
    print(df_sparse)

tfidf_vectoriser = TfidfVectorizer(analyzer=preprocess_text)
corpus_df = pd.DataFrame({'corpus': all_texts})
X_train_vectorised = tfidf_vectoriser.fit_transform(corpus_df['corpus'])

# Examen cualitativo con textos de prueba elegidos del propio dataset
print("── b) Examen cualitativo de descriptores TF-IDF ──")
test_samples = all_texts[6000:6003]
test_vectorised = tfidf_vectoriser.transform(test_samples)
display_tfidfs(test_vectorised)

In [ ]:
import pandas as pd
import torch
# Transformación a matriz
X_tfidf = pd.DataFrame.sparse.from_spmatrix(X_train_vectorised)
print(f"\n── c) Forma de la matriz TF-IDF: {X_tfidf.shape} ──")  # (N, vocab_size)


X = torch.tensor(X_tfidf.values, dtype=torch.float64)
t = torch.tensor(all_labels, dtype=torch.float64).unsqueeze(1)

In [ ]:
# Clasificación con regresión logística

X_train, X_test, t_train, t_test = train_test_split(
    X, t, test_size=0.2, random_state=42
)

model_tfidf = LogisticRegression(
    torch.zeros((X.shape[1], 1), dtype=torch.float64)
)
model_tfidf.train(X_train, t_train, steps=STEPS_LR)

print("\n── d) Resultados ──")
print(f"  Train accuracy: {model_tfidf.accuracy(X_train, t_train):.4f}")
print(f"  Test  accuracy: {model_tfidf.accuracy(X_test,  t_test):.4f}")


#Sección 2.2 - Regresión logística con vectores empotrados de BERT.

### a) Transformar entrada en vectores empotrados


In [ ]:



from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"


# ── Modelo 1: BERT en español (dccuchile) ─────────────────────────
tokenizerBERT = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
modelBERT = AutoModelForMaskedLM.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
modelBERT.eval()

# ── Modelo 2: BERTIN - RoBERTa en español ─────────────────────────
tokenizerRoBERTa = AutoTokenizer.from_pretrained("bertin-project/bertin-roberta-base-spanish")
modelRoBERTa = AutoModelForMaskedLM.from_pretrained("bertin-project/bertin-roberta-base-spanish")
modelRoBERTa.eval()

# ── Embedding mediante token CLS ──────────────────────────────────
def get_embeddings_batch(texts, model, tokenizer, batch_size=32, device="cpu"):
    model = model.to(device)
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True      # padea al más largo del batch
        ).to(device)

        with torch.no_grad():
            outputs = model.base_model(**inputs)

        # Token CLS de cada elemento del batch → shape (batch_size, 768)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(cls_embeddings.cpu())

    return torch.cat(all_embeddings, dim=0).numpy()  # (N, 768)


embInputsBERT    = get_embeddings_batch(all_texts, modelBERT,    tokenizerBERT,    device=device)
embInputsRoBERTa = get_embeddings_batch(all_texts, modelRoBERTa, tokenizerRoBERTa, device=device)

### b) Entrenar regresor logístico con el dataset empotrado

In [ ]:
from logistic_regression import LogisticRegression

STEPS_LR = 10000

#Con BERT
X = torch.tensor(embInputsBERT, dtype=torch.float64)
t = torch.tensor(all_labels, dtype=torch.float64).unsqueeze(1)  # (N,) → (N, 1)

modelB = LogisticRegression(torch.zeros((X.shape[1], 1), dtype=torch.float64))
modelB.train(X, t, STEPS_LR)

#Con RoBERTa
M = torch.tensor(embInputsRoBERTa, dtype=torch.float64)
y = torch.tensor(all_labels, dtype=torch.float64).unsqueeze(1)  # (N,) → (N, 1)

modelR = LogisticRegression(torch.zeros((X.shape[1], 1), dtype=torch.float64))
modelR.train(M, y, STEPS_LR)

### c) Evaluar con los mismos datos de entrenamiento la clasificación que hace entrenado

In [ ]:
accB = model.accuracy(X, t)
print(f"Accuracy BERT: {accB}")

accR = modelR.accuracy(M, y)
print(f"Accuracy RoBERTa: {accR}")

# ─────────────────────────────────────────────────────────────────
# Sección 2.3 — Modelos LLM para clasificación de complejidad
# Modelos:
#   1. Recognai/zeroshot_selectra_medium  (español, ELECTRA-based)
#   2. facebook/bart-large-mnli           (multilingüe, BART-based)
# ─────────────────────────────────────────────────────────────────

In [ ]:
# ── Sección 2.3 — LLMs ───────────────────────────────────────────
from llm_classifier import (
    load_classifiers, classify_all,
    show_examples, compute_llm_metrics, print_metrics_table
)

# Carga
classifiers = load_classifiers()

# 3 ejemplos cualitativos
ejemplos = [
    "El banco ofrece una cuenta de ahorros con interés anual del 3%.",
    "La tasa de rentabilidad ajustada por riesgo refleja la volatilidad "
    "implícita del subyacente en el mercado de derivados.",
    "Puedes guardar tu dinero en el banco y ganar un pequeño porcentaje extra.",
]
show_examples(classifiers, ejemplos)

# Dataset completo — all_texts y all_labels vienen de load_feina()
predictions = classify_all(classifiers, all_texts)
metrics     = compute_llm_metrics(predictions, all_labels)
print_metrics_table(metrics)